In [1]:
# Celda 1 · Cargar
import pandas as pd
pd.set_option("display.width", 200)

dep = pd.read_csv("datos/procesados/dengue_departamento_anio.csv")
sem = pd.read_csv("datos/procesados/dengue_nacional_semana.csv", parse_dates=["fecha_inicio"])
epi = pd.read_csv("datos/procesados/nino_episodios_enfen.csv", parse_dates=["inicio", "fin"])
hist = dep[dep["anio"] <= 2025]          # años completos
d26 = dep[dep["anio"] == 2026].copy()    # 2026, SE 1-42

In [2]:
# Celda 2 · Subpregunta 1: ¿dónde se concentra en 2026? (SE 1-42)
d26["pct_casos"] = (d26["casos_se42"] / d26["casos_se42"].sum() * 100).round(1)
cols = ["departamento", "casos_se42", "pct_casos", "incidencia_se42"]
print("Por casos:\n", d26.sort_values("casos_se42", ascending=False)[cols].head(8).to_string(index=False))
print("\nPor incidencia:\n", d26.sort_values("incidencia_se42", ascending=False)[cols].head(8).to_string(index=False))
top3 = d26.nlargest(3, "casos_se42")
print("\nTop 3 por casos:", ", ".join(top3["departamento"]),
      "→", round(top3["casos_se42"].sum() / d26["casos_se42"].sum() * 100, 1), "% del país")

Por casos:
 departamento  casos_se42  pct_casos  incidencia_se42
 LA LIBERTAD       16174       19.6            741.4
       PIURA       14761       17.9            672.4
  LAMBAYEQUE       11506       13.9            832.0
  SAN MARTIN        9297       11.3            962.5
        LIMA        7007        8.5             60.3
     UCAYALI        3844        4.7            579.2
      LORETO        3607        4.4            337.3
    AMAZONAS        3123        3.8            725.8

Por incidencia:
  departamento  casos_se42  pct_casos  incidencia_se42
MADRE DE DIOS        2745        3.3           1312.4
   SAN MARTIN        9297       11.3            962.5
       TUMBES        2289        2.8            836.6
   LAMBAYEQUE       11506       13.9            832.0
  LA LIBERTAD       16174       19.6            741.4
     AMAZONAS        3123        3.8            725.8
        PIURA       14761       17.9            672.4
      UCAYALI        3844        4.7            579.2

Top 3 

In [3]:
# Celda 3 · Subpregunta 3: ¿dónde queda 2026 entre todos los años, a la misma semana?
nac = dep.groupby("anio")[["casos_se42", "poblacion"]].sum()
nac["incidencia_se42"] = (nac["casos_se42"] / nac["poblacion"] * 100_000).round(1)
nac["puesto"] = nac["casos_se42"].rank(ascending=False).astype(int)
print(nac.sort_values("casos_se42", ascending=False).head(8))

prom = dep[dep["anio"].between(2021, 2025)].groupby("departamento")["incidencia_se42"].mean()
comp = d26.set_index("departamento")[["casos_se42", "incidencia_se42"]].join(prom.rename("prom_2021_2025"))
comp = comp[comp["prom_2021_2025"] > 0]
comp["veces_su_promedio"] = (comp["incidencia_se42"] / comp["prom_2021_2025"]).round(2)
print("\n2026 frente a su propio promedio 2021-2025 (SE 1-42):")
print(comp.sort_values("veces_su_promedio", ascending=False).head(10))

      casos_se42  poblacion  incidencia_se42  puesto
anio                                                
2024      262287   34038457            770.6       1
2023      242827   33725844            720.0       2
2026       82554   34660114            238.2       3
2017       66462   30973992            214.6       4
2022       53284   33396698            159.5       5
2021       34456   33035304            104.3       6
2015       34141   29964499            113.9       7
2025       32594   34350244             94.9       8

2026 frente a su propio promedio 2021-2025 (SE 1-42):
               casos_se42  incidencia_se42  prom_2021_2025  veces_su_promedio
departamento                                                                 
APURIMAC               10              2.4            0.62               3.87
PASCO                1121            429.0          173.54               2.47
MADRE DE DIOS        2745           1312.4          625.56               2.10
SAN MARTIN           9297

In [4]:
# Celda 4 · Subpregunta 2: ¿los picos coincidieron con los episodios de El Niño costero?
anual = hist.groupby("anio")["casos"].sum()
print("Años con más casos (2000-2025):\n", anual.sort_values(ascending=False).head(6))
print("\n2017 frente al promedio 2010-2016:", round(anual[2017] / anual.loc[2010:2016].mean(), 1), "veces")
print("2023 frente al promedio 2018-2022:", round(anual[2023] / anual.loc[2018:2022].mean(), 1), "veces")

fuera = pd.Series(True, index=sem.index)
for _, e in epi.iterrows():
    dentro = sem["fecha_inicio"].between(e["inicio"], e["fin"])
    fuera &= ~dentro
    print(f"\n{e['episodio']}: {int(sem.loc[dentro, 'casos'].sum()):,} casos en {dentro.sum()} semanas "
          f"→ promedio {sem.loc[dentro, 'casos'].mean():,.0f} por semana")
print(f"\nResto de semanas 2000-2025: promedio {sem.loc[fuera, 'casos'].mean():,.0f} por semana")

Años con más casos (2000-2025):
 anio
2024    271531.0
2023    256641.0
2017     68290.0
2022     63217.0
2020     47932.0
2021     44791.0
Name: casos, dtype: float64

2017 frente al promedio 2010-2016: 2.9 veces
2023 frente al promedio 2018-2022: 7.3 veces

El Niño costero 2017: 56,367 casos en 26 semanas → promedio 2,168 por semana

El Niño costero 2023-24: 374,892 casos en 55 semanas → promedio 6,816 por semana

Resto de semanas 2000-2025: promedio 496 por semana


In [5]:
# Celda 5 · ¿Qué departamentos fueron los más afectados en cada episodio? (incidencia anual)
for anios in [[2017], [2023, 2024]]:
    t = hist[hist["anio"].isin(anios)].groupby("departamento")["incidencia"].mean().round(1)
    print(f"\nIncidencia promedio {anios}:\n", t.sort_values(ascending=False).head(6).to_string())


Incidencia promedio [2017]:
 departamento
PIURA            2289.3
TUMBES           1757.7
ICA               489.3
MADRE DE DIOS     364.5
LA LIBERTAD       311.4
AYACUCHO          253.7

Incidencia promedio [2023, 2024]:
 departamento
TUMBES         3436.6
PIURA          2614.0
ICA            2242.1
LA LIBERTAD    1801.7
LAMBAYEQUE     1506.3
ANCASH         1267.8
